# Notebook 14 - Joins Deep Dive

**Dataset:** `samples.bakehouse` + `samples.tpch`  
**Difficulty:** Medium  
**Topics:** inner, left, right, full outer, left semi, left anti, cross join, broadcast hint, handling duplicate columns after join


In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder.getOrCreate()

customers    = spark.table("samples.bakehouse.sales_customers")
transactions = spark.table("samples.bakehouse.sales_transactions")
franchises   = spark.table("samples.bakehouse.sales_franchises")
tpch_orders  = spark.table("samples.tpch.orders")
tpch_customer = spark.table("samples.tpch.customer")
tpch_nation  = spark.table("samples.tpch.nation")
tpch_region  = spark.table("samples.tpch.region")


## Problem 1 - Inner vs Outer Join Comparison

Join `sales_customers` (left) to `sales_transactions` (right) on `customerID`.  
Run both inner join and left join, then build a summary DataFrame comparing row counts.  
This shows that left join includes customers with no transactions.  
**Columns:** `join_type`, `row_count`


In [ ]:
result_1 = None
# YOUR CODE HERE


In [ ]:
# ── Tests for Problem 1 ─────────────────
assert result_1 is not None, "result_1 is None"
assert hasattr(result_1, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'join_type' in cols, "Missing: join_type"
assert 'row_count' in cols, "Missing: row_count"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_1.count()
assert cnt == 2, f"Expected exactly 2 rows (inner + left), got {cnt}"
# Left join should always have >= rows than inner join
rows = {r['join_type']: r['row_count'] for r in result_1.collect()}
join_types = list(rows.keys())
assert len(join_types) == 2, "Expected 2 join type rows"
counts = list(rows.values())
assert max(counts) >= min(counts), "Left join should have >= rows than inner join"
print(f"Problem 1 passed ✓  ({cnt} rows)")


## Problem 2 - Left Semi-Join (EXISTS)

Use `how="left_semi"` to find customers who have made at least one transaction - without duplicating customer rows.  
**Columns:** `customerID`, `first_name`, `last_name`, `country`


In [ ]:
result_2 = None
# YOUR CODE HERE


In [ ]:
# ── Tests for Problem 2 ─────────────────
assert result_2 is not None, "result_2 is None"
assert hasattr(result_2, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'customerid' in cols, "Missing: customerID"
assert 'first_name' in cols, "Missing: first_name"
assert 'last_name' in cols, "Missing: last_name"
assert 'country' in cols, "Missing: country"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_2.count()
assert cnt > 0, "Got 0 rows"
# No duplicate customerIDs (semi join should deduplicate)
distinct_cnt = result_2.select("customerID").distinct().count()
assert distinct_cnt == cnt, "customerID should be unique - semi join should not produce duplicates"
# All returned customers must appear in transactions
txn_customers = transactions.select("customerID").distinct()
unmatched = result_2.join(txn_customers, on="customerID", how="left_anti")
assert unmatched.count() == 0, "Some customers in result have no transactions"
print(f"Problem 2 passed ✓  ({cnt} rows)")


## Problem 3 - Left Anti-Join (NOT EXISTS)

Use `how="left_anti"` to find customers who have **never** made a transaction.  
**Columns:** `customerID`, `first_name`, `last_name`, `country`


In [ ]:
result_3 = None
# YOUR CODE HERE


In [ ]:
# ── Tests for Problem 3 ─────────────────
assert result_3 is not None, "result_3 is None"
assert hasattr(result_3, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
assert 'customerid' in cols, "Missing: customerID"
assert 'first_name' in cols, "Missing: first_name"
assert 'last_name' in cols, "Missing: last_name"
assert 'country' in cols, "Missing: country"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_3.count()
# semi + anti should add up to total customers
total_customers = customers.count()
semi_cnt = result_2.count()
assert cnt + semi_cnt == total_customers, f"semi ({semi_cnt}) + anti ({cnt}) should equal total customers ({total_customers})"
# None of the anti-join customers should appear in transactions
txn_customers = transactions.select("customerID").distinct()
overlap = result_3.join(txn_customers, on="customerID", how="inner")
assert overlap.count() == 0, "Anti-join customers should not appear in transactions"
print(f"Problem 3 passed ✓  ({cnt} rows)")


## Problem 4 - Full Outer Join

Join `tpch.customer` FULL OUTER with `tpch.orders` on `c_custkey = o_custkey`.  
Show how many customers have no orders (`o_orderkey` is null) and how many orders have no matching customer (`c_custkey` is null after join).  
**Columns:** `category`, `count`


In [ ]:
result_4 = None
# YOUR CODE HERE


In [ ]:
# ── Tests for Problem 4 ─────────────────
assert result_4 is not None, "result_4 is None"
assert hasattr(result_4, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
assert 'category' in cols, "Missing: category"
assert 'count' in cols, "Missing: count"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_4.count()
assert cnt > 0, "Got 0 rows"
assert cnt <= 3, "Expected at most 3 categories"
# All counts should be non-negative
neg = result_4.filter(F.col("count") < 0)
assert neg.count() == 0, "count values must be non-negative"
print(f"Problem 4 passed ✓  ({cnt} rows)")


## Problem 5 - Broadcast Hint for Small Table

Join `tpch.nation` (25 rows - small) with `tpch.customer` (large) using `F.broadcast(df_nation)` to avoid a shuffle.  
**Why it helps:** the small table is replicated to each executor, eliminating the expensive shuffle of the large table.  
**Columns:** `c_name`, `c_mktsegment`, `n_name`  
Limit to 1000 rows.


In [ ]:
result_5 = None
# YOUR CODE HERE


In [ ]:
# ── Tests for Problem 5 ─────────────────
assert result_5 is not None, "result_5 is None"
assert hasattr(result_5, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
assert 'c_name' in cols, "Missing: c_name"
assert 'c_mktsegment' in cols, "Missing: c_mktsegment"
assert 'n_name' in cols, "Missing: n_name"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_5.count()
assert cnt > 0, "Got 0 rows"
assert cnt <= 1000, "Expected at most 1000 rows (limited)"
# n_name should be non-null (broadcast join matched)
nulls = result_5.filter(F.col("n_name").isNull())
assert nulls.count() == 0, "n_name should not be null after inner join with nation"
print(f"Problem 5 passed ✓  ({cnt} rows)")


## Problem 6 - Handle Duplicate Column Names After Join

Join `sales_customers` (has `city`) with `sales_transactions` on `customerID`.  
Both DataFrames share column names - handle the ambiguity by aliasing DataFrames before joining and selecting with qualified names.  
**Columns:** `transactionID`, `customer_city`, `product`, `totalPrice`


In [ ]:
result_6 = None
# YOUR CODE HERE


In [ ]:
# ── Tests for Problem 6 ─────────────────
assert result_6 is not None, "result_6 is None"
assert hasattr(result_6, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_6.columns]
assert 'transactionid' in cols, "Missing: transactionID"
assert 'customer_city' in cols, "Missing: customer_city"
assert 'product' in cols, "Missing: product"
assert 'totalprice' in cols, "Missing: totalPrice"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_6.count()
assert cnt > 0, "Got 0 rows"
# customer_city should not be null
nulls = result_6.filter(F.col("customer_city").isNull())
assert nulls.count() == 0, "customer_city should not be null"
print(f"Problem 6 passed ✓  ({cnt} rows)")


## Problem 7 - Cross Join (Cartesian Product)

Cross join `tpch.region` (5 rows) with a manually created small DataFrame of quarters `['Q1','Q2','Q3','Q4']`.  
Build the quarters DataFrame with `spark.createDataFrame([(q,) for q in ['Q1','Q2','Q3','Q4']], ['quarter'])`.  
**Columns:** `r_name`, `quarter`  
Should produce exactly 20 rows (5 × 4).


In [ ]:
result_7 = None
# YOUR CODE HERE


In [ ]:
# ── Tests for Problem 7 ─────────────────
assert result_7 is not None, "result_7 is None"
assert hasattr(result_7, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_7.columns]
assert 'r_name' in cols, "Missing: r_name"
assert 'quarter' in cols, "Missing: quarter"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_7.count()
assert cnt == 20, f"Cross join of 5 regions x 4 quarters should produce 20 rows, got {cnt}"
# Distinct quarters should be Q1-Q4
quarters = sorted([r['quarter'] for r in result_7.select("quarter").distinct().collect()])
assert quarters == ['Q1', 'Q2', 'Q3', 'Q4'], f"Expected Q1-Q4, got {quarters}"
# Distinct regions should be 5
assert result_7.select("r_name").distinct().count() == 5, "Expected 5 distinct regions"
print(f"Problem 7 passed ✓  ({cnt} rows)")
